# Kaggle submission notebook

## 1. Notebook setup
### 1.1. Imports

In [1]:
# Standard library
import json

# Third party
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import KNNImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler

### 1.2. Run configuration

In [2]:
SAMPLE = 0.10      # Fraction of data to use for training
IMPUTE = True      # True = do imputation, False = load imputation results from disk
KNN_NEIGHBORS = 3  # Number of neighbors to use for KNN imputation
CV_SCORE_FOLDS = 5 # Number of cross-validation folds to use for scoring

### 1.3. Data loading

#### 1.3.1. Training data

In [ ]:
# Load raw data
train_df = pd.read_csv('../data/train.csv')

# Take sample
n = int(len(train_df) * SAMPLE)
train_df = train_df.sample(n=n)

# Preserve label column
training_label = train_df['health_condition']

# Remove id and label columns
train_df.drop(['id', 'health_condition'], axis=1, inplace=True)
train_df.head().T

,544212,482071,567542,23373,558232,107771,411187,673356,448364,210322,...,483226,200369,614615,394576,456286,668794,583960,305746,28608,377949
sleep_duration,8.22,9.17,7.16,7.28,5.85,NaN,6.0,NaN,5.84,8.96,...,NaN,NaN,6.58,6.36,7.15,7.88,6.17,4.71,7.22,6.55
heart_rate,76.5,66.8,66.4,76.2,80.7,79.1,67.5,69.0,85.7,74.6,...,79.0,77.1,66.0,61.4,81.4,61.9,77.9,85.3,81.3,NaN
bmi,18.74,25.09,25.27,19.1,24.93,21.46,27.08,21.17,27.2,25.07,...,22.61,23.79,23.07,19.76,23.05,23.68,22.14,24.88,24.01,25.04
calorie_expenditure,2070.0,2395.0,1795.0,2333.0,2037.0,2105.0,2166.0,2842.0,2325.0,2117.0,...,2484.0,2390.0,NaN,1739.0,2428.0,2662.0,2115.0,2283.0,NaN,1818.0
step_count,12073.0,12456.0,6856.0,7262.0,2384.0,9592.0,9382.0,13129.0,6795.0,2878.0,...,1675.0,3695.0,12333.0,9338.0,5352.0,11209.0,3999.0,3649.0,11594.0,6409.0


#### 1.3.2. Testing data

In [ ]:
# Load raw data
test_df = pd.read_csv('../data/test.csv')

# Preserve id column
testing_ids = test_df['id']

# Drop id column
test_df.drop('id', axis=1, inplace=True)
test_df.head().T

,0,1,2,3,4,5,6,7,8,9,...,295743,295744,295745,295746,295747,295748,295749,295750,295751,295752
sleep_duration,5.35,NaN,6.68,7.13,5.49,8.51,6.21,7.17,7.95,NaN,...,4.84,6.8,NaN,NaN,7.35,7.13,6.25,NaN,7.25,4.61
heart_rate,64.9,83.1,59.7,78.5,77.7,72.0,65.6,65.7,61.1,73.7,...,71.1,77.0,74.9,72.2,77.1,77.5,69.5,77.0,66.3,85.1
bmi,23.48,22.42,24.14,26.26,23.29,21.96,25.02,NaN,22.16,21.89,...,21.28,24.05,23.25,22.38,23.05,22.2,20.93,23.62,21.47,23.79
calorie_expenditure,2745.0,1773.0,3040.0,2494.0,1828.0,2449.0,1788.0,NaN,NaN,2121.0,...,2448.0,2444.0,2484.0,1415.0,1494.0,2221.0,2242.0,2111.0,2009.0,2456.0
step_count,14167.0,6801.0,13250.0,6331.0,13894.0,7232.0,4884.0,14082.0,3228.0,8961.0,...,6447.0,2454.0,1746.0,3187.0,2913.0,13981.0,2370.0,7647.0,4672.0,14195.0


#### 1.3.3. Dataset metadata

Load feature definitions from disk. Treat our one high cardinality discrete feature `step_count` as continuous.

In [5]:
# Load dataset metadata
with open("../data/schema.json", "r") as f:
    metadata = json.load(f)

# Extract feature lists
features = metadata['features']
categorical_features = metadata['categorical_features']
continuous_features = metadata['continuous_features']
high_cardinality_feature = metadata['high_cardinality_feature']
label = metadata['label']

# Treat 'step_count' like a continuous feature
continuous_features += [high_cardinality_feature]

print(f'Categorical features: {", ".join(categorical_features)}')
print(f'Continuous features: {", ".join(continuous_features)}')
print(f'Label: {label}')

Categorical features: diet_type, stress_level, sleep_quality, physical_activity_level, smoking_alcohol, gender
Continuous features: sleep_duration, heart_rate, bmi, calorie_expenditure, step_count, exercise_duration, water_intake, step_count
Label: health_condition


## 2. Data preprocessing

Minimal data pre-processing: standard scaling, one-hot encoding for categorical features, ordinal encoding for the label and KNN imputation for missing values in continuous features, with a missing indicator.

### 2.1. Feature scaling

Standard scale the continuous features so they are weighted equally in by the KNN imputer.

In [ ]:
# Scikit-learn StandardScaler for continuous features
scaler = StandardScaler()

# Adapt to the training data continuous features
scaler.fit(train_df[continuous_features])

# Transform both training and testing data continuous features
train_df[continuous_features] = scaler.transform(train_df[continuous_features])
test_df[continuous_features] = scaler.transform(test_df[continuous_features])

### 2.2. Feature encoding

#### 2.2.1. One-hot encode categorical features

In [7]:
# Scikit-learn one-hot encoder
feature_encoder = OneHotEncoder(sparse_output=False, drop='first')

# Adapt to training data categorical features
result = feature_encoder.fit(train_df[categorical_features])

# Transform training and testing data
encoded_train_features = feature_encoder.transform(train_df[categorical_features])
encoded_test_features = feature_encoder.transform(test_df[categorical_features])

#### 2.2.2. Add encoded features back to DataFrame

In [8]:
# Convert encoded features to DataFrame
encoded_train_df = pd.DataFrame(encoded_train_features, columns=feature_encoder.get_feature_names_out())
encoded_test_df = pd.DataFrame(encoded_test_features, columns=feature_encoder.get_feature_names_out())

# Replace original categorical features with encoded features
train_df = pd.concat(
    [train_df.drop(columns=categorical_features).reset_index(drop=True),
    encoded_train_df.reset_index(drop=True)],
    axis=1
)

test_df = pd.concat(
    [test_df.drop(columns=categorical_features).reset_index(drop=True),
    encoded_test_df.reset_index(drop=True)],
    axis=1
)

# Update feature list
features = train_df.columns

train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69008 entries, 0 to 69007
Data columns (total 25 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   sleep_duration                     61313 non-null  float64
 1   heart_rate                         68230 non-null  float64
 2   bmi                                67564 non-null  float64
 3   calorie_expenditure                63676 non-null  float64
 4   step_count                         67557 non-null  float64
 5   exercise_duration                  68292 non-null  float64
 6   water_intake                       64618 non-null  float64
 7   diet_type_non-veg                  69008 non-null  float64
 8   diet_type_veg                      69008 non-null  float64
 9   diet_type_nan                      69008 non-null  float64
 10  stress_level_low                   69008 non-null  float64
 11  stress_level_medium                69008 non-null  flo

### 2.3. Label encoding

In [9]:
# Scikit-learn label encoder
label_encoder = LabelEncoder()

# Convert string training labels to integer representation
training_label = label_encoder.fit_transform(training_label)

### 2.3. Imputation

One-hot encoding handles missing data in the categorical features by including a class for 'missing'. We want to use all features for input to the imputer to get the best possible estimates for the missing values. But if we pass in all features and let imputer automatically add the missing indicator feature, it will be added to all of the features - including the one-hot encoded features which already have a missing category. To avoid this, we manually construct missing indicator features for only the continuous features and then run the imputer with the built in missing indicator off.

Imputation of the full dataset takes approximately 4 hours. The computationally expensive (and long) imputation can be skipped using the `IMPUTE` boolean flag.

#### 2.3.1. Create boolean 'missing' indicator features

In [10]:
if IMPUTE:

    # Create column names for the new indicator features
    indicator_features = [f'{feature}_missing' for feature in train_df[continuous_features].columns]

    # Create indicator features from boolean mask
    training_missing = train_df[continuous_features].isna().astype('int')
    training_missing.columns = indicator_features

    testing_missing = test_df[continuous_features].isna().astype('int')
    testing_missing.columns = indicator_features

    # Add indicator features to DataFrames
    train_df = pd.concat(
        [train_df.reset_index(drop=True),
        training_missing.reset_index(drop=True)],
        axis=1
    )

    test_df = pd.concat(
        [test_df.reset_index(drop=True),
        testing_missing.reset_index(drop=True)],
        axis=1
    )

    train_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69008 entries, 0 to 69007
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   sleep_duration                     61313 non-null  float64
 1   heart_rate                         68230 non-null  float64
 2   bmi                                67564 non-null  float64
 3   calorie_expenditure                63676 non-null  float64
 4   step_count                         67557 non-null  float64
 5   exercise_duration                  68292 non-null  float64
 6   water_intake                       64618 non-null  float64
 7   diet_type_non-veg                  69008 non-null  float64
 8   diet_type_veg                      69008 non-null  float64
 9   diet_type_nan                      69008 non-null  float64
 10  stress_level_low                   69008 non-null  float64
 11  stress_level_medium                69008 non-null  flo

#### 2.3.1. Adapt KNN imputer on training data

In [ ]:
if IMPUTE: # Do the imputation

    # Distance weighted Scikit-learn KNN imputer
    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS, weights='distance')

    # Fit imputer on training data
    imputer.fit(train_df)

CPU times: user 15.6 ms, sys: 47 μs, total: 15.6 ms
Wall time: 14.6 ms


#### 2.3.2. Impute training & testing data

In [12]:
%%time

if IMPUTE: # Do the imputation

    # Impute both datasets
    train_df = pd.DataFrame(
        imputer.transform(train_df),
        columns=imputer.get_feature_names_out()
    )

    test_df = pd.DataFrame(
        imputer.transform(test_df),
        columns=imputer.get_feature_names_out()
    )

CPU times: user 19min 47s, sys: 12min 18s, total: 32min 5s
Wall time: 10min 13s


#### 2.3.3. Save (or load) imputed DataFrames

In [13]:
if IMPUTE: # Do the imputation

    # Save expensive imputation computation result
    train_df.to_csv(f"../data/train_imputed_n{n}.csv", index=False)
    test_df.to_csv(f"../data/test_imputed_n{n}.csv", index=False)

else: # Load prior imputation results

    train_df = pd.read_csv(f"../data/train_imputed_n{n}.csv")
    test_df = pd.read_csv(f"../data/test_imputed_n{n}.csv")

train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69008 entries, 0 to 69007
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   sleep_duration                     69008 non-null  float64
 1   heart_rate                         69008 non-null  float64
 2   bmi                                69008 non-null  float64
 3   calorie_expenditure                69008 non-null  float64
 4   step_count                         69008 non-null  float64
 5   exercise_duration                  69008 non-null  float64
 6   water_intake                       69008 non-null  float64
 7   diet_type_non-veg                  69008 non-null  float64
 8   diet_type_veg                      69008 non-null  float64
 9   diet_type_nan                      69008 non-null  float64
 10  stress_level_low                   69008 non-null  float64
 11  stress_level_medium                69008 non-null  flo

## 3. Gradient boosting classifier

Use an 'off-th-shelf' gradient boosting classifier from Scikit-learn to predict `health_condition` for the test set.

### 3.1. Performance estimation

In [14]:
# Cross-validation fold generator
cv = KFold(n_splits=CV_SCORE_FOLDS, shuffle=True, random_state=315)

# Scikit-learn gradient boosting classifier with defaults
gbc = GradientBoostingClassifier()

# Use cross validation to evaluate the model
scores = cross_val_score(
    gbc,
    train_df,
    training_label, 
    cv=CV_SCORE_FOLDS,
    scoring='balanced_accuracy',
)

# Get 95% confidence interval bounds
upper_bound = scores.mean() + 1.96 * scores.std() / (len(scores) ** 0.5)
lower_bound = scores.mean() - 1.96 * scores.std() / (len(scores) ** 0.5)

# Show results
print(f"Cross-validation balanced accuracy scores: {''.join([f'{x:.3f}, ' for x in scores])}\n")
print(f"Mean cross-validation balanced accuracy score: {scores.mean():.3f}")
print(f"95% confidence interval: ({lower_bound:.3f}, {upper_bound:.3f})")

Cross-validation balanced accuracy scores: 0.848, 0.869, 0.859, 0.871, 0.867, 

Mean cross-validation balanced accuracy score: 0.863
95% confidence interval: (0.856, 0.870)


### 3.2. Model training

In [15]:
# Fit the model on the entire training set
result = gbc.fit(train_df, training_label)

### 3.3. Make test set predictions for submission

In [16]:
# Make predictions on the test set
predictions = gbc.predict(test_df)

# Un-encode predictions to get string labels back
predictions = label_encoder.inverse_transform(predictions)

# Convert to dataframe
submission_df = pd.DataFrame({'id': testing_ids, 'health_condition': predictions})
submission_df.head()

,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [17]:
# Save predictions to a CSV file
submission_df.to_csv('../data/submission.csv', index=False)